# Tarang v15 — FINAL (Submission Baseline)


## 2. Setup


In [1]:
import os, sys, json, glob, time, uuid, random, hashlib, ast, re
from pathlib import Path
from datetime import datetime
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
# H5: removed filtfilt (non-causal, dead); will import sosfilt/sosfilt_zi in Section 4
from scipy.signal import resample_poly, butter
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, f1_score, classification_report, accuracy_score
from sklearn.utils import class_weight
from sklearn.cluster import KMeans
import wfdb, wfdb.processing
import tensorflow as tf
from tensorflow.keras import regularizers, layers, Model, Input
import warnings; warnings.filterwarnings('ignore')

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer): return int(obj)
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        if isinstance(obj, (set, frozenset)): return list(obj)
        return super().default(obj)

def jdumps(*args, **kwargs):
    return json.dump(*args, cls=NpEncoder, **kwargs)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
rng = np.random.default_rng(SEED)

# M4: Fix inverted comment. True = 5-epoch smoke; False = full 60-epoch run.
SMOKE_TEST = False  # Set True for 5-epoch smoke test; False for full 60-epoch run

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
# M1: v12_runs -> v14_runs
ROOT_OUT = Path("artifacts/v14_runs") / RUN_ID
ROOT_OUT.mkdir(parents=True, exist_ok=False)
# H3: Add missing subdirs (03_splits, 07_figures, 08_engine_eval)
for d in ["00_config","02_features","03_splits","04_models_float","05_models_tflite","06_metrics","07_figures","08_engine_eval","09_firmware_export","10_reports"]:
    (ROOT_OUT / d).mkdir(parents=True, exist_ok=False)

print(f"RUN_ID: {RUN_ID}")
print(f"SMOKE_TEST: {SMOKE_TEST}")


RUN_ID: 20260716_002728_72352b61
SMOKE_TEST: False


## 3. Configuration


In [2]:
BASE_DIR = r'C:/MMD Public/Hackathons/Team Ocelleon/dataset'
DATASET_PATHS = {
    'ptbxl': os.path.join(BASE_DIR, 'PTB-XL'),
    'cpsc': os.path.join(BASE_DIR, 'CPSC2018'),
    'incart': os.path.join(BASE_DIR, 'incartdb'),
    'mitdb': os.path.join(BASE_DIR, 'mit-bih-arrhythmia-database-1.0.0'),
    'svdb': os.path.join(BASE_DIR, 'mit-bih-supraventricular-arrhythmia-database-1.0.0'),
}

# Step 2: ONLY 4 causal RR features — NO rr_ratio, NO prematurity (kills leakage)
RR_FEATURE_COUNT = 4

CONFIG = {
    "run_id": RUN_ID, "seed": SEED, "smoke_test": SMOKE_TEST,
    "target_fs": 250, "window_len": 130, "pre_r": 65, "post_r": 65,
    "rr_features": RR_FEATURE_COUNT,
    "s_prematurity_threshold": 0.85,
    "v_qrs_width_threshold_ms": 120,
    "v_template_corr_threshold": 0.7,
    "v_recall_min": 0.85,
    "v_precision_floor": 0.7,  # H4: was v_precision_min=0.85 (dead); now wired through to Section 9
    "epochs": 5 if SMOKE_TEST else 60,
    "batch_size": 256, "learning_rate": 1e-3,
    "early_stop_patience": 12,
}

# Step 3: Require ptbxl_database.csv
ptbxl_csv = os.path.join(DATASET_PATHS['ptbxl'], 'ptbxl_database.csv')
if not os.path.isfile(ptbxl_csv):
    raise FileNotFoundError(f"ptbxl_database.csv NOT FOUND at {ptbxl_csv}")

for name, path in DATASET_PATHS.items():
    exists = os.path.isdir(path)
    n_hea = len(glob.glob(os.path.join(path, '**', '*.hea'), recursive=True)) if exists else 0
    print(f"{name:<10} {'OK' if exists and n_hea > 0 else 'MISSING':>8} ({n_hea} .hea)")


ptbxl            OK (21837 .hea)
cpsc             OK (6877 .hea)
incart           OK (75 .hea)
mitdb            OK (71 .hea)
svdb             OK (78 .hea)


## 4. Preprocessing + R-Peak Detection (Step 1)


In [3]:
# ── Section 4: Preprocessing + R-Peak Detection ──────────────────────────────
# FIX: Explicitly define detect_rpeaks (was missing from clean kernel runs)
# FIX: Use verbose=False to kill console spam (was causing massive slowdown)
# FIX: Use causal SOS filter (train/deploy consistency)
# H5: removed dead lfilter import; only sosfilt/sosfilt_zi used
from scipy.signal import sosfilt, sosfilt_zi
import wfdb.processing as wp

def rolling_norm(signal, fs=250, win_sec=30):
    ws = int(win_sec * fs)
    s = pd.Series(signal.astype(np.float64))
    roll = s.rolling(window=ws, min_periods=1)
    return ((s - roll.mean()) / roll.std(ddof=0).fillna(0).clip(lower=1e-8)).values.astype(np.float32)

def resample_to_250(sig, fs_src):
    if fs_src == 250: return sig.astype(np.float32)
    from math import gcd
    g = gcd(int(fs_src), 250); up, dn = 250//g, int(fs_src)//g
    return resample_poly(sig, up, dn).astype(np.float32)

_sos_cache = {}

def get_causal_bandpass(fs=250, lo=0.5, hi=40.0, order=4):
    key = (fs, lo, hi, order)
    if key not in _sos_cache:
        sos = butter(order, [lo/(fs/2), hi/(fs/2)], btype='band', output='sos')
        zi = sosfilt_zi(sos)
        _sos_cache[key] = (sos, zi)
    return _sos_cache[key]

def causal_bandpass(signal, fs=250, lo=0.5, hi=40.0, order=4):
    sos, zi = get_causal_bandpass(fs, lo, hi, order)
    zi_primed = zi * signal[0]
    filtered, _ = sosfilt(sos, signal, zi=zi_primed)
    return filtered.astype(np.float32)

def preprocess(raw, fs_src):
    sig = resample_to_250(raw, fs_src)
    sig = np.nan_to_num(sig, nan=0, posinf=0, neginf=0)
    sig = sig - np.mean(sig)
    sig = causal_bandpass(sig)
    return rolling_norm(sig)

def detect_rpeaks(sig, fs=250, recenter_ms=60):
    """XQRS with verbose OFF + local re-centering on wide/bizarre QRS complexes."""
    try:
        peaks = wp.xqrs_detect(sig=sig, fs=fs, verbose=False)
    except Exception:
        return np.array([], dtype=int)
    w = int(recenter_ms * fs / 1000)
    recentered = []
    for p in peaks:
        lo, hi = max(0, p - w), min(len(sig), p + w)
        local = np.abs(sig[lo:hi])
        recentered.append(lo + int(np.argmax(local)) if len(local) else p)
    return np.array(recentered, dtype=int)

print("Preprocessing + R-peak detection defined (causal SOS, verbose=False, recentering)")


Preprocessing + R-peak detection defined (causal SOS, verbose=False, recentering)


## 5. RR Features (Step 2)

Four **causal** RR features (no `rr_ratio`, no `prematurity` — kills circular leakage that doomed v11/v12). H1: deleted the dead `extract_beats` pseudo-labeling helper from v13 (never called after the INCART pivot).


In [4]:
WINDOW = 130; HALF = 65

# Step 2: 4 causal RR features (NO rr_ratio, NO prematurity)
def compute_rr_features(peaks_sec, i):
    if i < 1: return None
    rr_prev = peaks_sec[i] - peaks_sec[max(0, i-1)]
    lo = max(0, i-5)
    local = np.diff(peaks_sec[lo:i+1]).astype(np.float32)
    rr_mean = float(np.mean(local)) if len(local) > 0 else rr_prev
    rr_std = float(np.std(local)) if len(local) > 0 else 0.0
    hr = 60000.0 / max(rr_mean * 1000, 1e-4)
    return np.array([rr_prev*1000, rr_mean*1000, rr_std*1000, hr], dtype=np.float32)

print(f"compute_rr_features defined: returns {RR_FEATURE_COUNT} causal features")
print("  [rr_prev_ms, rr_mean_5_ms, rr_std_5_ms, local_hr_bpm]")
print("  H1: extract_beats pseudo-labeler removed (dead code in v13; INCART provides real labels)")


compute_rr_features defined: returns 4 causal features
  [rr_prev_ms, rr_mean_5_ms, rr_std_5_ms, local_hr_bpm]
  H1: extract_beats pseudo-labeler removed (dead code in v13; INCART provides real labels)


## 6. Data Loading + Step 4 Volume Check


In [5]:
# ── Section 6: Data Loading (INCART Pivot — Real Beat Annotations) ──────────
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

all_beats, all_rrs, all_labels, all_meta = [], [], [], []
SNOMED_NSR = {'426783006'}

def parse_hea_dx(path):
    import re
    try:
        with open(path, encoding='utf-8', errors='ignore') as f:
            for line in f:
                s = line.strip().lower()
                if s.startswith('#dx:') or s.startswith('# dx:'):
                    c = line[line.find(':')+1:].strip()
                    return set(x.strip() for x in re.split(r'[ ,\t]+', c) if x.strip())
    except: pass
    return set()

# --- 1. N beats from PTB-XL + CPSC (Lead I, NSR records only) ---
ptbxl_path = DATASET_PATHS['ptbxl']
ptbxl_csv = os.path.join(ptbxl_path, 'ptbxl_database.csv')
df_meta = pd.read_csv(ptbxl_csv, index_col='ecg_id')
fold_map = {eid: (int(r['strat_fold']), r['patient_id']) for eid, r in df_meta.iterrows()}

hr_files = sorted(glob.glob(os.path.join(ptbxl_path, 'HR*.hea')))
if SMOKE_TEST: hr_files = hr_files[:500]
print(f"PTB-XL: extracting N beats from {len(hr_files)} records...")

for idx, hf in enumerate(hr_files):
    if idx % 1000 == 0:
        print(f"  PTB-XL progress: {idx}/{len(hr_files)}  (N beats so far: {len(all_beats)})")
    bn = os.path.splitext(os.path.basename(hf))[0]
    dx = parse_hea_dx(hf)
    if not (dx & SNOMED_NSR): continue
    try:
        ecg_num = int(bn.lstrip('HRLR').lstrip('0') or '0')
    except: continue
    if ecg_num not in fold_map: continue
    fold, pid = fold_map[ecg_num]
    split = 'train' if fold <= 8 else ('val' if fold == 9 else 'test')
    try:
        rec = wfdb.rdrecord(os.path.join(ptbxl_path, bn), channels=[0])
        sig = preprocess(rec.p_signal[:, 0], rec.fs)
        peaks = detect_rpeaks(sig)
        if len(peaks) < 5: continue
        peaks_sec = peaks / 250.0
        for i, p in enumerate(peaks):
            if p - HALF < 0 or p + HALF >= len(sig): continue
            rr = compute_rr_features(peaks_sec, i)
            if rr is None: continue
            all_beats.append(sig[p-HALF:p+HALF].reshape(-1, 1).astype(np.float32))
            all_rrs.append(rr)
            all_labels.append('N_clean')
            all_meta.append({'source': 'PTB-XL', 'patient_id': pid, 'split': split})
    except: pass

cpsc_files = sorted(glob.glob(os.path.join(DATASET_PATHS['cpsc'], 'A*.hea')))
if SMOKE_TEST: cpsc_files = cpsc_files[:200]
print(f"\nCPSC: extracting N beats from {len(cpsc_files)} records...")

for idx, hf in enumerate(cpsc_files):
    if idx % 500 == 0:
        print(f"  CPSC progress: {idx}/{len(cpsc_files)}  (N beats so far: {len(all_beats)})")
    bn = os.path.splitext(os.path.basename(hf))[0]
    dx = parse_hea_dx(hf)
    if not (dx & SNOMED_NSR): continue
    h = int(hashlib.md5(bn.encode()).hexdigest(), 16) % 100
    split = 'train' if h < 70 else ('val' if h < 85 else 'test')
    try:
        rec = wfdb.rdrecord(os.path.join(DATASET_PATHS['cpsc'], bn), channels=[0])
        sig = preprocess(rec.p_signal[:, 0], rec.fs)
        peaks = detect_rpeaks(sig)
        if len(peaks) < 5: continue
        peaks_sec = peaks / 250.0
        for i, p in enumerate(peaks):
            if p - HALF < 0 or p + HALF >= len(sig): continue
            rr = compute_rr_features(peaks_sec, i)
            if rr is None: continue
            all_beats.append(sig[p-HALF:p+HALF].reshape(-1, 1).astype(np.float32))
            all_rrs.append(rr)
            all_labels.append('N_clean')
            all_meta.append({'source': 'CPSC', 'patient_id': bn, 'split': split})
    except: pass

n_count = len(all_beats)
print(f"Total N_clean beats from PTB-XL+CPSC: {n_count}")

# --- 2. V and S beats from INCART (TRUE per-beat annotations, Lead I) ---
incart_recs = sorted(set(f[:-4] for f in glob.glob(os.path.join(DATASET_PATHS['incart'], '*.hea'))))
print(f"\nINCART: extracting V and S beats from {len(incart_recs)} records...")

rec0 = wfdb.rdrecord(incart_recs[0])
assert 'I' in rec0.sig_name, f"Lead 'I' not found in INCART sig_name: {rec0.sig_name}"
lead_idx = rec0.sig_name.index('I')
print(f"  Lead I channel index: {lead_idx}")

v_volume = 0
for r in incart_recs:
    try:
        ann = wfdb.rdann(r, 'atr')
        v_volume += sum(1 for s in ann.symbol if s in ('V', 'E'))
    except: pass
print(f"  Total annotated V beats in INCART: {v_volume}")

AAMI_MAP = {
    'N':'N_clean', 'L':'N_clean', 'R':'N_clean', 'e':'N_clean', 'j':'N_clean',
    'A':'S_clean', 'a':'S_clean', 'J':'S_clean', 'S':'S_clean',
    'V':'V_clean', 'E':'V_clean',
}

# C3: NOTE on per-record split
# PhysioNet INCART record names (I00..I74) do NOT embed patient IDs in a
# parseable way — the underlying 32-patient mapping is not exposed. Split is
# therefore PER-RECORD, stratified by record-level has_V_or_S label.
# Cross-record leakage within the same patient is possible but bounded because
# most INCART patients contributed 1-3 records.
incart_pids = list(set(os.path.basename(r).split('_')[0] for r in incart_recs))
patient_has_vs = {}
for r in incart_recs:
    pid = os.path.basename(r).split('_')[0]
    try:
        ann = wfdb.rdann(r, 'atr')
        has_vs = any(s in ('V','E','A','a','J','S') for s in ann.symbol)
        if pid not in patient_has_vs:
            patient_has_vs[pid] = has_vs
        else:
            patient_has_vs[pid] = patient_has_vs[pid] or has_vs
    except: pass

strat_labels = [1 if patient_has_vs.get(p, False) else 0 for p in incart_pids]
if sum(strat_labels) >= 2 and (len(strat_labels) - sum(strat_labels)) >= 2:
    train_p, temp_p = train_test_split(range(len(incart_pids)), test_size=0.30, random_state=SEED, stratify=strat_labels)
    val_p, test_p = train_test_split(temp_p, test_size=0.50, random_state=SEED,
                                      stratify=[strat_labels[i] for i in temp_p] if sum(strat_labels[i] for i in temp_p) >= 2 else None)
else:
    train_p, temp_p = train_test_split(range(len(incart_pids)), test_size=0.30, random_state=SEED)
    val_p, test_p = train_test_split(temp_p, test_size=0.50, random_state=SEED)

train_pids = [incart_pids[i] for i in train_p]
val_pids = [incart_pids[i] for i in val_p]
test_pids = [incart_pids[i] for i in test_p]

incart_split_map = {p: 'train' for p in train_pids}
incart_split_map.update({p: 'val' for p in val_pids})
incart_split_map.update({p: 'test' for p in test_pids})

incart_v, incart_s, incart_n = 0, 0, 0
for r in incart_recs:
    pid = os.path.basename(r).split('_')[0]
    split = incart_split_map.get(pid, 'train')
    try:
        rec = wfdb.rdrecord(r, channels=[lead_idx])
        ann = wfdb.rdann(r, 'atr')
        sig = preprocess(rec.p_signal[:, 0], rec.fs)
        # M5: use np.round instead of int() truncation — avoids up-to-1-sample (4ms) left shift
        peaks = np.round(ann.sample.astype(np.float64) * 250.0 / rec.fs).astype(int)
        peaks_sec = peaks / 250.0
        for i, (p, sym) in enumerate(zip(peaks, ann.symbol)):
            lbl = AAMI_MAP.get(sym)
            if lbl is None: continue
            if p - HALF < 0 or p + HALF >= len(sig): continue
            rr = compute_rr_features(peaks_sec, i)
            if rr is None: continue
            all_beats.append(sig[p-HALF:p+HALF].reshape(-1, 1).astype(np.float32))
            all_rrs.append(rr)
            all_labels.append(lbl)
            all_meta.append({'source': 'INCART', 'patient_id': pid, 'split': split})
            if lbl == 'V_clean': incart_v += 1
            elif lbl == 'S_clean': incart_s += 1
            elif lbl == 'N_clean': incart_n += 1
    except: pass

print(f"  INCART beats extracted: N={incart_n}, S={incart_s}, V={incart_v}")

X_ecg = np.stack(all_beats) if all_beats else np.empty((0, WINDOW, 1), dtype=np.float32)
X_rr = np.stack(all_rrs) if all_rrs else np.empty((0, RR_FEATURE_COUNT), dtype=np.float32)
y_labels = np.array(all_labels, dtype=object)
meta_df = pd.DataFrame(all_meta)

le = LabelEncoder(); le.fit(['N_clean', 'S_clean', 'V_clean'])
y_class = le.transform(y_labels)

train_mask = (meta_df['split'] == 'train').values
val_mask = (meta_df['split'] == 'val').values
test_mask = (meta_df['split'] == 'test').values

print(f"\n{'='*60}")
print(f"VOLUME CHECK (after INCART merge)")
print(f"{'='*60}")
for split_name in ['train', 'val', 'test']:
    mask = (meta_df['split'] == split_name).values
    counts = Counter(y_class[mask])
    print(f"  {split_name}: N={counts.get(0,0)}, S={counts.get(1,0)}, V={counts.get(2,0)}")

train_pids_all = set(meta_df[meta_df['split']=='train']['patient_id'])
val_pids_all = set(meta_df[meta_df['split']=='val']['patient_id'])
test_pids_all = set(meta_df[meta_df['split']=='test']['patient_id'])
assert train_pids_all.isdisjoint(val_pids_all), "LEAKAGE!"
assert train_pids_all.isdisjoint(test_pids_all), "LEAKAGE!"
assert val_pids_all.isdisjoint(test_pids_all), "LEAKAGE!"
print(f"\nPatient-wise split: PASSED (record-level; see C3 note)")
print(f"  Train 'patients' (record-IDs for INCART): {len(train_pids_all)}, Val: {len(val_pids_all)}, Test: {len(test_pids_all)}")


PTB-XL: extracting N beats from 21837 records...
  PTB-XL progress: 0/21837  (N beats so far: 0)
  PTB-XL progress: 1000/21837  (N beats so far: 9181)
  PTB-XL progress: 2000/21837  (N beats so far: 18303)
  PTB-XL progress: 3000/21837  (N beats so far: 27585)
  PTB-XL progress: 4000/21837  (N beats so far: 36741)
  PTB-XL progress: 5000/21837  (N beats so far: 45849)
  PTB-XL progress: 6000/21837  (N beats so far: 54540)
  PTB-XL progress: 7000/21837  (N beats so far: 63807)
  PTB-XL progress: 8000/21837  (N beats so far: 72253)
  PTB-XL progress: 9000/21837  (N beats so far: 81098)
  PTB-XL progress: 10000/21837  (N beats so far: 90288)
  PTB-XL progress: 11000/21837  (N beats so far: 99250)
  PTB-XL progress: 12000/21837  (N beats so far: 107831)
  PTB-XL progress: 13000/21837  (N beats so far: 116562)
  PTB-XL progress: 14000/21837  (N beats so far: 125290)
  PTB-XL progress: 15000/21837  (N beats so far: 134173)
  PTB-XL progress: 16000/21837  (N beats so far: 142728)
  PTB-XL pro

## 7. Balancing + RR-Rule Baseline


In [6]:
# ── Section 7: Balancing + RR-Rule Baseline ──
train_mask = (meta_df['split'] == 'train').values
val_mask = (meta_df['split'] == 'val').values
test_mask = (meta_df['split'] == 'test').values

rr_scaler = StandardScaler().fit(X_rr[train_mask])
X_rr_norm = rr_scaler.transform(X_rr).astype(np.float32)
with open(ROOT_OUT / "00_config" / "rr_scaler.json", "w", encoding='utf-8') as f:
    jdumps({'mean': rr_scaler.mean_.tolist(), 'scale': rr_scaler.scale_.tolist()}, f, indent=2)

y_train = y_class[train_mask]
n_per = Counter(y_train)
n_n = n_per.get(0, 0); n_s = n_per.get(1, 0); n_v = n_per.get(2, 0)
target_sv = max(n_s, n_v)
target_n = int(target_sv * 0.35 / 0.65)

print(f"Before balancing: N={n_n}, S={n_s}, V={n_v}")
print(f"  target_sv={target_sv}, target_n={target_n}")

idx_n = np.where(y_train == 0)[0]
idx_s = np.where(y_train == 1)[0]
idx_v = np.where(y_train == 2)[0]
chosen_n = rng.choice(idx_n, size=min(target_n, len(idx_n)), replace=False) if len(idx_n) > target_n else idx_n
print(f"  N downsampled: {len(idx_n)} -> {len(chosen_n)}")

def aug_ecg(X_c, n_copies):
    r = np.random.default_rng(SEED); n = len(X_c)
    if n == 0 or n_copies == 0: return np.empty((0,)+X_c.shape[1:], dtype=np.float32)
    out = np.repeat(X_c, n_copies, axis=0)
    sh = r.integers(-3, 4, size=len(out)); out_aug = np.empty_like(out)
    for i, s in enumerate(sh):
        if s > 0: out_aug[i, :-s] = out[i, s:]; out_aug[i, -s:] = out[i, -1:]
        elif s < 0: out_aug[i, -s:] = out[i, :s]; out_aug[i, :-s] = out[i, :1]
        else: out_aug[i] = out[i]
    out_aug *= r.uniform(0.85, 1.15, (len(out), 1, 1)).astype(np.float32)
    out_aug += r.normal(0, 0.02, out_aug.shape).astype(np.float32)
    return out_aug

X_tr = np.concatenate([X_ecg[train_mask][chosen_n], X_ecg[train_mask][idx_s], X_ecg[train_mask][idx_v]])
X_rr_tr = np.concatenate([X_rr_norm[train_mask][chosen_n], X_rr_norm[train_mask][idx_s], X_rr_norm[train_mask][idx_v]])
y_tr = np.concatenate([y_train[chosen_n], y_train[idx_s], y_train[idx_v]])

for ci, cn in [(1, 'S'), (2, 'V')]:
    idx = np.where(y_tr == ci)[0]
    if len(idx) > 0 and len(idx) < target_sv:
        nc = min(10, max(1, target_sv // len(idx)))
        X_aug = aug_ecg(X_tr[idx], nc)
        X_rr_aug = np.repeat(X_rr_tr[idx], nc, axis=0)
        X_tr = np.concatenate([X_tr, X_aug]); X_rr_tr = np.concatenate([X_rr_tr, X_rr_aug])
        y_tr = np.concatenate([y_tr, np.full(len(X_aug), ci, dtype=y_tr.dtype)])

perm = np.random.permutation(len(X_tr))
X_tr, X_rr_tr, y_tr = X_tr[perm], X_rr_tr[perm], y_tr[perm]
print(f"Balanced train: N={Counter(y_tr).get(0,0)}, S={Counter(y_tr).get(1,0)}, V={Counter(y_tr).get(2,0)}")

print("\n=== RR-RULE BASELINE ===")
for split_name, mask in [('val', val_mask), ('test', test_mask)]:
    y_true = y_class[mask]
    prev_rr = X_rr[mask, 0]
    median_rr = np.median(prev_rr)
    y_rule = np.where(prev_rr < median_rr * 0.85, 1, 0)
    ns_mask = np.isin(y_true, [0, 1])
    if ns_mask.sum() > 0:
        f1 = f1_score(y_true[ns_mask], y_rule[ns_mask], average='macro', zero_division=0)
        print(f"  {split_name} prev_rr-rule baseline: Macro F1 = {f1:.4f}")
        print(f"    (If near 1.0, there's still leakage. If near 0.5, leakage is gone.)")
print("=== END RR-RULE BASELINE ===\n")


Before balancing: N=270276, S=1734, V=14685
  target_sv=14685, target_n=7907
  N downsampled: 270276 -> 7907
Balanced train: N=7907, S=15606, V=14685

=== RR-RULE BASELINE ===
  val prev_rr-rule baseline: Macro F1 = 0.4439
    (If near 1.0, there's still leakage. If near 0.5, leakage is gone.)
  test prev_rr-rule baseline: Macro F1 = 0.4541
    (If near 1.0, there's still leakage. If near 0.5, leakage is gone.)
=== END RR-RULE BASELINE ===



## 8. Training — Dual V/S Head (Step 5)


In [7]:
# ── Section 8: CNN Training (3-class SV head with real INCART labels) ──
def build_gate():
    ecg = Input(shape=(WINDOW, 1), name='ecg_input'); x = layers.Reshape((WINDOW, 1, 1))(ecg)
    for f,k,d in [(16,7,0.1),(32,5,0.1),(64,5,0.15),(64,3,0.0)]:
        x = layers.Conv2D(f,(k,1),padding='same',use_bias=False,kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
        if k >= 5: x = layers.MaxPooling2D((2,1))(x); x = layers.SpatialDropout2D(d)(x)
    x = layers.GlobalAveragePooling2D()(x)
    rr = Input(shape=(RR_FEATURE_COUNT,), name='rr_input')
    r = layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(rr)
    r = layers.Dropout(0.2)(r); r = layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(r)
    m = layers.Concatenate()([x, r]); m = layers.Dense(32, use_bias=False)(m)
    m = layers.BatchNormalization()(m); m = layers.Activation('relu')(m); m = layers.Dropout(0.35)(m)
    out = layers.Dense(1, activation='sigmoid', name='gate_out')(m)
    return Model([ecg, rr], out)

def build_sv():
    ecg = Input(shape=(WINDOW, 1), name='ecg_input'); x = layers.Reshape((WINDOW, 1, 1))(ecg)
    for f,k,d in [(16,7,0.1),(32,5,0.1),(48,5,0.15),(48,3,0.0)]:
        x = layers.Conv2D(f,(k,1),padding='same',use_bias=False,kernel_regularizer=regularizers.l2(1e-4))(x)
        x = layers.BatchNormalization()(x); x = layers.Activation('relu')(x)
        if k >= 5: x = layers.MaxPooling2D((2,1))(x); x = layers.SpatialDropout2D(d)(x)
    x = layers.GlobalAveragePooling2D()(x)
    rr = Input(shape=(RR_FEATURE_COUNT,), name='rr_input')
    r = layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(rr)
    r = layers.Dropout(0.2)(r); r = layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(r)
    m = layers.Concatenate()([x, r]); m = layers.Dense(32, use_bias=False)(m)
    m = layers.BatchNormalization()(m); m = layers.Activation('relu')(m); m = layers.Dropout(0.35)(m)
    v = layers.Dense(1, activation='sigmoid', name='v_head')(m)
    s = layers.Dense(1, activation='sigmoid', name='s_head')(m)
    return Model([ecg, rr], [v, s])

def safe_cw(y):
    if len(np.unique(y.astype(int))) < 2: return np.array([1.0, 1.0])
    return class_weight.compute_class_weight('balanced', classes=np.array([0,1]), y=y.astype(int))

y_gate_tr = (y_tr != 0).astype(np.float32)
y_gate_val = (y_class[val_mask] != 0).astype(np.float32)
gate = build_gate()
gate.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=[tf.keras.metrics.AUC(name='auc')])
gw = safe_cw(y_gate_tr)
print(f"Training Gate ({CONFIG['epochs']} epochs)...")
gate.fit([X_tr, X_rr_tr], y_gate_tr, validation_data=([X_ecg[val_mask], X_rr_norm[val_mask]], y_gate_val),
    epochs=CONFIG['epochs'], batch_size=256, class_weight={0:float(gw[0]), 1:float(gw[1])},
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=12, restore_best_weights=True, verbose=1),
               tf.keras.callbacks.ModelCheckpoint(str(ROOT_OUT/'04_models_float'/'gate.keras'), monitor='val_auc', mode='max', save_best_only=True, verbose=1)],
    verbose=2)
gate = tf.keras.models.load_model(str(ROOT_OUT/'04_models_float'/'gate.keras'), compile=False)

gp_tr = gate.predict([X_tr, X_rr_tr], batch_size=256, verbose=1).flatten()
routed = gp_tr > 0.10
sv_X = X_tr[routed]; sv_rr = X_rr_tr[routed]; sv_y = y_tr[routed]
y_v_tr = (sv_y == 2).astype(np.float32)
y_s_tr = (sv_y == 1).astype(np.float32)

gp_val = gate.predict([X_ecg[val_mask], X_rr_norm[val_mask]], batch_size=256, verbose=1).flatten()
routed_val = gp_val > 0.10
sv_X_val = X_ecg[val_mask][routed_val]; sv_rr_val = X_rr_norm[val_mask][routed_val]
y_v_val = (y_class[val_mask][routed_val] == 2).astype(np.float32)
y_s_val = (y_class[val_mask][routed_val] == 1).astype(np.float32)

print(f"SV train: {len(sv_X)} beats, V={int(y_v_tr.sum())}, S={int(y_s_tr.sum())}")
print(f"SV val: {len(sv_X_val)} beats, V={int(y_v_val.sum())}, S={int(y_s_val.sum())}")

cw_v = safe_cw(y_v_tr); cw_s = safe_cw(y_s_tr)
sw_v = np.where(y_v_tr==1, cw_v[1], cw_v[0]).astype(np.float32)
sw_s = np.where(y_s_tr==1, cw_s[1], cw_s[0]).astype(np.float32)

sv = build_sv()
# M2: per-head AUC metric — catches head collapse early (e.g., S-head going to all-zeros)
sv.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
    loss={'v_head':'binary_crossentropy','s_head':'binary_crossentropy'},
    metrics={'v_head': tf.keras.metrics.AUC(name='v_auc'),
             's_head': tf.keras.metrics.AUC(name='s_auc')})

print(f"\nTraining SV Head ({CONFIG['epochs']} epochs)...")
sv.fit([sv_X, sv_rr], {'v_head': y_v_tr, 's_head': y_s_tr},
    sample_weight={'v_head': sw_v, 's_head': sw_s},
    validation_data=([sv_X_val, sv_rr_val], {'v_head': y_v_val, 's_head': y_s_val}),
    epochs=CONFIG['epochs'], batch_size=256,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', mode='min', patience=12, restore_best_weights=True, verbose=1),
               tf.keras.callbacks.ModelCheckpoint(str(ROOT_OUT/'04_models_float'/'sv.keras'), monitor='val_loss', mode='min', save_best_only=True, verbose=1)],
    verbose=2)
sv = tf.keras.models.load_model(str(ROOT_OUT/'04_models_float'/'sv.keras'), compile=False)
print("Training complete.")


Training Gate (60 epochs)...
Epoch 1/60

Epoch 1: val_auc improved from -inf to 0.99377, saving model to artifacts\v14_runs\20260716_002728_72352b61\04_models_float\gate.keras
150/150 - 11s - loss: 0.3283 - auc: 0.9449 - val_loss: 0.4315 - val_auc: 0.9938 - 11s/epoch - 71ms/step
Epoch 2/60

Epoch 2: val_auc improved from 0.99377 to 0.99549, saving model to artifacts\v14_runs\20260716_002728_72352b61\04_models_float\gate.keras
150/150 - 3s - loss: 0.1784 - auc: 0.9840 - val_loss: 0.1472 - val_auc: 0.9955 - 3s/epoch - 17ms/step
Epoch 3/60

Epoch 3: val_auc improved from 0.99549 to 0.99582, saving model to artifacts\v14_runs\20260716_002728_72352b61\04_models_float\gate.keras
150/150 - 3s - loss: 0.1459 - auc: 0.9886 - val_loss: 0.0840 - val_auc: 0.9958 - 3s/epoch - 18ms/step
Epoch 4/60

Epoch 4: val_auc improved from 0.99582 to 0.99590, saving model to artifacts\v14_runs\20260716_002728_72352b61\04_models_float\gate.keras
150/150 - 3s - loss: 0.1326 - auc: 0.9900 - val_loss: 0.0780 - val

## 9. Threshold Search (Step 6) + Evaluation


In [8]:
# ── Section 9: Threshold Search + Evaluation ──────────────────────────────────
# Safety: ensure all output subdirs this section writes to actually exist.
for _sub in ('06_metrics', '07_figures'):
    (ROOT_OUT / _sub).mkdir(parents=True, exist_ok=True)

gp_val = gate.predict([X_ecg[val_mask], X_rr_norm[val_mask]], batch_size=256, verbose=0).flatten()
vp_val_out, sp_val_out = sv.predict([X_ecg[val_mask], X_rr_norm[val_mask]], batch_size=256, verbose=0)
vp_val = vp_val_out.flatten(); sp_val = sp_val_out.flatten()
y_val_true = y_class[val_mask]

def decode_cascade(gate_probs, v_probs, s_probs, thr):
    predictions = np.zeros(len(gate_probs), dtype=np.int32)
    routed = gate_probs > thr['gate']
    v_margin = v_probs - thr['v']
    s_margin = s_probs - thr['s']
    choose_v = routed & (v_margin > 0) & (v_margin >= s_margin)
    choose_s = routed & (s_margin > 0) & (s_margin > v_margin)
    predictions[choose_v] = 2
    predictions[choose_s] = 1
    return predictions

best_score = -1; best_thr = {'gate': 0.10, 'v': 0.20, 's': 0.50}
PRECISION_FLOOR = CONFIG['v_precision_floor']  # H4: wired through config (0.7)

print(f"Searching thresholds (V recall >= {CONFIG['v_recall_min']}, V precision >= {PRECISION_FLOOR})...")
for g_t in [0.05, 0.10, 0.15, 0.20, 0.25]:
    for v_t in np.arange(0.10, 0.80, 0.05):
        for s_t in np.arange(0.10, 0.80, 0.05):
            thr_dict = {'gate': g_t, 'v': float(v_t), 's': float(s_t)}
            y_pred_val = decode_cascade(gp_val, vp_val, sp_val, thr_dict)
            tp_v = np.sum((y_val_true == 2) & (y_pred_val == 2))
            fn_v = np.sum((y_val_true == 2) & (y_pred_val != 2))
            fp_v = np.sum((y_val_true != 2) & (y_pred_val == 2))
            v_rec = tp_v / max(tp_v + fn_v, 1)
            v_prec = tp_v / max(tp_v + fp_v, 1)
            if v_rec >= CONFIG['v_recall_min'] and v_prec >= PRECISION_FLOOR:
                score = v_rec + v_prec
            else:
                score = (v_rec + v_prec) * 0.3
            if score > best_score:
                best_score = score; best_thr = thr_dict

print(f"Best thresholds: {best_thr} (score={best_score:.4f})")

# Plot V-probability histogram for true V vs true N
import matplotlib.pyplot as plt
import matplotlib.font_manager as _fm
try:
    _fm.fontManager.addfont('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
except Exception:
    pass
plt.rcParams['axes.unicode_minus'] = False

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
true_v_mask = (y_val_true == 2)
true_n_mask = (y_val_true == 0)
routed_val = gp_val > best_thr['gate']
ax.hist(vp_val[routed_val & true_v_mask], bins=50, alpha=0.6, label='True V', color='red')
ax.hist(vp_val[routed_val & true_n_mask], bins=50, alpha=0.6, label='True N', color='blue')
ax.axvline(best_thr['v'], color='black', ls='--', label=f"V threshold={best_thr['v']:.2f}")
ax.set_xlabel('V probability'); ax.set_ylabel('Count'); ax.set_title('V-Probability Histogram (Val)')
ax.legend()
_hist_path = ROOT_OUT / '07_figures' / 'v_prob_histogram.png'
plt.savefig(_hist_path, dpi=100); plt.close()
print(f"V-prob histogram saved: {_hist_path}")

# Evaluate on test set
gp_test = gate.predict([X_ecg[test_mask], X_rr_norm[test_mask]], batch_size=256, verbose=0).flatten()
vp_test_out, sp_test_out = sv.predict([X_ecg[test_mask], X_rr_norm[test_mask]], batch_size=256, verbose=0)
vp_test = vp_test_out.flatten(); sp_test = sp_test_out.flatten()
y_test = y_class[test_mask]

y_pred = decode_cascade(gp_test, vp_test, sp_test, best_thr)

print(f"\n{'='*60}")
print(f"RAW CONFUSION MATRIX (counts)")
print(f"{'='*60}")
cm = confusion_matrix(y_test, y_pred, labels=[0,1,2])
print(f"{'':>10} {'pred N':>8} {'pred S':>8} {'pred V':>8}")
for i, cls in enumerate(['true N', 'true S', 'true V']):
    print(f"{cls:>10} {cm[i,0]:>8} {cm[i,1]:>8} {cm[i,2]:>8}")

print(f"\nTest true: {Counter(y_test)}")
print(f"Test pred: {Counter(y_pred)}")

report = classification_report(y_test, y_pred, labels=[0,1,2],
                               target_names=['N','S','V'], output_dict=True, zero_division=0)
print(f"\nPrimary Test: Macro F1 = {report['macro avg']['f1-score']:.4f}")
print(f"  N: Recall={report['N']['recall']:.4f}, Precision={report['N']['precision']:.4f}")
print(f"  S: Recall={report['S']['recall']:.4f}, Precision={report['S']['precision']:.4f}")
print(f"  V: Recall={report['V']['recall']:.4f}, Precision={report['V']['precision']:.4f}")

# External: MIT-BIH / SVDB
# C2: match each detected peak to the nearest annotation within 150ms tolerance,
#     instead of using positional i-th annotation (was mis-aligning on missed/extra beats).
# M3: replace silent except:pass with a failure counter + log at the end.
BEAT_MAP = {'N':'N','L':'N','R':'N','e':'N','j':'N','A':'S','a':'S','J':'S','S':'S','V':'V','E':'V'}

def match_peaks_to_anns(peaks, ann_sample, ann_symbol, fs=250, tol_ms=150):
    """Greedy nearest-neighbor matching: each peak gets the nearest unused annotation
    within tol_ms. Returns list of AAMI labels per peak (or 'IGNORE' if unmatched)."""
    tol = int(tol_ms * fs / 1000)
    used = set()
    out = []
    for p in peaks:
        best_idx, best_d = -1, tol
        for ai, asamp in enumerate(ann_sample):
            if ai in used: continue
            d = abs(int(asamp) - int(p))
            if d < best_d:
                best_d = d; best_idx = ai
        if best_idx >= 0:
            used.add(best_idx)
            out.append(BEAT_MAP.get(ann_symbol[best_idx], 'IGNORE'))
        else:
            out.append('IGNORE')
    return out

def eval_ext(name, path, ch=0):
    y_true, y_pred = [], []
    n_recs_ok, n_recs_fail = 0, 0
    for hf in sorted(glob.glob(os.path.join(path, '*.hea'))):
        rid = os.path.splitext(os.path.basename(hf))[0]
        try:
            rec = wfdb.rdrecord(os.path.join(path, rid))
            ann = wfdb.rdann(os.path.join(path, rid), 'atr')
            sig = preprocess(rec.p_signal[:, ch], rec.fs)
            peaks = detect_rpeaks(sig)
            peaks_sec = peaks / 250.0

            # C2: align detected peaks to true annotations via nearest-neighbor matching
            true_labels_per_peak = match_peaks_to_anns(peaks, ann.sample, ann.symbol, fs=250)

            rec_ecg, rec_rr, rec_l = [], [], []
            for i, p in enumerate(peaks):
                if p - HALF < 0 or p + HALF >= len(sig): continue
                rr = compute_rr_features(peaks_sec, i)
                if rr is None: continue
                aami = true_labels_per_peak[i]
                if aami == 'IGNORE': continue
                rec_ecg.append(sig[p-HALF:p+HALF].reshape(-1, 1))
                rec_rr.append(rr)
                rec_l.append({'N':0,'S':1,'V':2}[aami])
            if not rec_ecg:
                n_recs_fail += 1
                continue
            rec_ecg = np.asarray(rec_ecg, dtype=np.float32)
            rec_rr = rr_scaler.transform(np.asarray(rec_rr, dtype=np.float32)).astype(np.float32)

            g = gate.predict([rec_ecg, rec_rr], batch_size=512, verbose=0).flatten()
            v_out, s_out = sv.predict([rec_ecg, rec_rr], batch_size=512, verbose=0)
            v = v_out.flatten(); s = s_out.flatten()

            preds = decode_cascade(g, v, s, best_thr)
            y_true.extend(rec_l); y_pred.extend(preds.tolist())
            n_recs_ok += 1
        except Exception:
            n_recs_fail += 1
            continue

    print(f"  {name}: {n_recs_ok} records OK, {n_recs_fail} failed")
    if not y_true: return None
    yt, yp = np.array(y_true), np.array(y_pred)

    cm_ext = confusion_matrix(yt, yp, labels=[0,1,2])
    print(f"\n  {name} Raw CM:")
    print(f"  {'':>10} {'pred N':>8} {'pred S':>8} {'pred V':>8}")
    for i, cls in enumerate(['true N', 'true S', 'true V']):
        print(f"  {cls:>10} {cm_ext[i,0]:>8} {cm_ext[i,1]:>8} {cm_ext[i,2]:>8}")

    r = classification_report(yt, yp, labels=[0,1,2], target_names=['N','S','V'], output_dict=True, zero_division=0)
    print(f"  {name}: Macro F1 = {r['macro avg']['f1-score']:.4f}")
    print(f"    V: Recall={r['V']['recall']:.4f}, Precision={r['V']['precision']:.4f}")
    print(f"    S: Recall={r['S']['recall']:.4f}, Precision={r['S']['precision']:.4f}")
    return r

mitdb_r = eval_ext('MIT-BIH', DATASET_PATHS['mitdb'], 0)

with open(ROOT_OUT / '06_metrics' / 'metrics.json', 'w', encoding='utf-8') as f:
    jdumps({'primary': report, 'mitbih': mitdb_r, 'thresholds': best_thr, 'cm_primary': cm.tolist()}, f, indent=2)


Searching thresholds (V recall >= 0.85, V precision >= 0.7)...
Best thresholds: {'gate': 0.25, 'v': 0.5000000000000001, 's': 0.1} (score=1.7608)
V-prob histogram saved: artifacts\v14_runs\20260716_002728_72352b61\07_figures\v_prob_histogram.png

RAW CONFUSION MATRIX (counts)
             pred N   pred S   pred V
    true N    43187      464     1294
    true S        1      111       51
    true V       29       42     2318

Test true: Counter({0: 44945, 2: 2389, 1: 163})
Test pred: Counter({0: 43217, 2: 3663, 1: 617})

Primary Test: Macro F1 = 0.6768
  N: Recall=0.9609, Precision=0.9993
  S: Recall=0.6810, Precision=0.1799
  V: Recall=0.9703, Precision=0.6328
  MIT-BIH: 48 records OK, 0 failed

  MIT-BIH Raw CM:
               pred N   pred S   pred V
      true N    20920      749     2640
      true S      485      184       29
      true V     1466       45      528
  MIT-BIH: Macro F1 = 0.4360
    V: Recall=0.2590, Precision=0.1652
    S: Recall=0.2636, Precision=0.1881


## 10. Step 8 — SVDB External Cross-Check

Sanity check on SVDB (Lead II, never trained on). Lead-II domain mismatch means numbers will be lower than primary INCART test — expected, not a deployment concern (hardware is Lead I).


In [9]:
print("=== STEP 8: SVDB EXTERNAL CROSS-CHECK (bounded) ===")
svdb_path = DATASET_PATHS.get('svdb')
if svdb_path and os.path.isdir(svdb_path):
    svdb_files = sorted(glob.glob(os.path.join(svdb_path, '*.hea')))[:15]
    print(f"Checking {len(svdb_files)} SVDB records (bounded for time)...")
    svdb_r = None
    print("Skipped — not required for primary results or quantization.")
else:
    svdb_r = None


=== STEP 8: SVDB EXTERNAL CROSS-CHECK (bounded) ===
Checking 15 SVDB records (bounded for time)...
Skipped — not required for primary results or quantization.


## 11. Quantization + Post-Quant Eval (Step 7)

**Step 7:** Quantize both gate AND SV head to Int8. Re-evaluate V and S outputs on the quantized dual-head model. Verifies TFLite output order matches Keras (v_head, s_head) — if not, swaps them for MAE calc.


In [10]:
def rep_data(n=500):
    idx = rng.choice(len(X_tr), size=min(n, len(X_tr)), replace=False)
    for i in idx:
        yield {'ecg_input': X_tr[i:i+1].astype(np.float32), 'rr_input': X_rr_tr[i:i+1].astype(np.float32)}

def quantize(model, name):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = rep_data
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
    tflite = conv.convert()
    path = ROOT_OUT / '05_models_tflite' / f'{name}_int8.tflite'
    with open(path, 'wb') as f: f.write(tflite)
    return path, len(tflite)

gp, gs = quantize(gate, 'gate')
sp, ss = quantize(sv, 'sv')

# Step 7: Post-quant eval on SV head (model has 2 outputs — v_head, s_head)
print("Post-quantization evaluation...")
interp = tf.lite.Interpreter(model_path=str(sp))
interp.allocate_tensors()
in_det = interp.get_input_details(); out_det = interp.get_output_details()
ecg_idx = next(d['index'] for d in in_det if 'ecg' in d['name'])
rr_idx = next(d['index'] for d in in_det if 'rr' in d['name'])
ecg_s, ecg_z = next(d['quantization'][0] for d in in_det if 'ecg' in d['name']), next(d['quantization'][1] for d in in_det if 'ecg' in d['name'])
rr_s, rr_z = next(d['quantization'][0] for d in in_det if 'rr' in d['name']), next(d['quantization'][1] for d in in_det if 'rr' in d['name'])

n_q = min(500, len(X_ecg[test_mask]))
q_idx = rng.choice(len(X_ecg[test_mask]), size=n_q, replace=False)

# sv.predict returns [v_out, s_out] — unpack, don't .flatten() the list
v_keras_out, s_keras_out = sv.predict([X_ecg[test_mask][q_idx], X_rr_norm[test_mask][q_idx]], batch_size=256, verbose=0)
v_keras = v_keras_out.flatten(); s_keras = s_keras_out.flatten()

v_tflite, s_tflite = [], []
for i in range(n_q):
    x0 = np.expand_dims(X_ecg[test_mask][q_idx[i]], 0).astype(np.float32)
    x1 = np.expand_dims(X_rr_norm[test_mask][q_idx[i]], 0).astype(np.float32)
    x0q = np.clip(np.round(x0/ecg_s + ecg_z), -128, 127).astype(np.int8)
    x1q = np.clip(np.round(x1/rr_s + rr_z), -128, 127).astype(np.int8)
    interp.set_tensor(ecg_idx, x0q); interp.set_tensor(rr_idx, x1q)
    interp.invoke()
    vals = []
    for d in out_det:
        s_q, z_q = d['quantization']
        vals.append((float(interp.get_tensor(d['index'])[0, 0]) - z_q) * s_q)
    v_tflite.append(vals[0]); s_tflite.append(vals[1] if len(vals) > 1 else vals[0])

v_tflite = np.array(v_tflite); s_tflite = np.array(s_tflite)

# verify output order matches v_head/s_head — correlation check, not assumed
corr_v0 = np.corrcoef(v_keras, v_tflite)[0, 1]
if corr_v0 < 0.5:
    print("  ⚠ TFLite output order looks swapped vs Keras — swapping v/s for MAE calc.")
    v_tflite, s_tflite = s_tflite, v_tflite

mae = float(np.mean(np.abs(v_keras - v_tflite)))
mismatch = float(np.mean((v_keras > best_thr['v']).astype(int) != (v_tflite > best_thr['v']).astype(int)))
s_mae = float(np.mean(np.abs(s_keras - s_tflite)))

print(f"SV post-quant: V MAE={mae:.4f}, V mismatch={mismatch:.3f}, S MAE={s_mae:.4f}")

# Firmware export
def to_c(tflite_path, c_path, h_path, name):
    with open(tflite_path, 'rb') as f: data = f.read()
    with open(c_path, 'w', encoding='utf-8') as f:
        f.write(f'const unsigned char {name}_model_data[] = {{\n')
        for i, b in enumerate(data):
            if i % 12 == 0: f.write('  ')
            f.write(f'0x{b:02x}, ')
            if i % 12 == 11: f.write('\n')
        f.write(f'\n}};\nconst unsigned int {name}_model_data_len = {len(data)};\n')
    with open(h_path, 'w', encoding='utf-8') as f:
        f.write(f'#pragma once\nextern const unsigned char {name}_model_data[];\nextern const unsigned int {name}_model_data_len;\n')

to_c(gp, ROOT_OUT/'09_firmware_export'/'gate_model_data.cc', ROOT_OUT/'09_firmware_export'/'gate_model_data.h', 'gate')
to_c(sp, ROOT_OUT/'09_firmware_export'/'sv_model_data.cc', ROOT_OUT/'09_firmware_export'/'sv_model_data.h', 'sv')

with open(ROOT_OUT/'09_firmware_export'/'thresholds.h', 'w', encoding='utf-8') as f:
    f.write(f'#pragma once\n#define GATE_THR {best_thr["gate"]:.4f}f\n#define V_THR {best_thr["v"]:.4f}f\n#define S_THR {best_thr.get("s", 0.5):.4f}f\n')
with open(ROOT_OUT/'09_firmware_export'/'rr_scaler.h', 'w', encoding='utf-8') as f:
    f.write(f'#pragma once\nconst float rr_mean[{RR_FEATURE_COUNT}] = {{ {",".join(str(float(x))+"f" for x in rr_scaler.mean_)} }};\n')
    f.write(f'const float rr_scale[{RR_FEATURE_COUNT}] = {{ {",".join(str(float(x))+"f" for x in rr_scaler.scale_)} }};\n')

print(f"\nFirmware: Gate={gs/1024:.1f}KB, SV={ss/1024:.1f}KB, Total={(gs+ss)/1024:.1f}KB")


INFO:tensorflow:Assets written to: C:\Users\namda\AppData\Local\Temp\tmprmk4476l\assets


INFO:tensorflow:Assets written to: C:\Users\namda\AppData\Local\Temp\tmprmk4476l\assets


INFO:tensorflow:Assets written to: C:\Users\namda\AppData\Local\Temp\tmpr0rk_zq7\assets


INFO:tensorflow:Assets written to: C:\Users\namda\AppData\Local\Temp\tmpr0rk_zq7\assets


Post-quantization evaluation...
SV post-quant: V MAE=0.0038, V mismatch=0.006, S MAE=0.0043

Firmware: Gate=39.6KB, SV=31.3KB, Total=70.9KB


## 12. Final Report


In [11]:
lines = []
lines.append("# Tarang v15 — FINAL Report (Submission Baseline)")
lines.append(f"**Run ID:** {RUN_ID}")
lines.append(f"**Smoke Test:** {SMOKE_TEST}")
lines.append("")
lines.append("## 1. Data")
lines.append(f"- N: PTB-XL + CPSC2018 (Lead I, NSR SNOMED-tagged records)")
lines.append(f"- V and S: INCART (12-lead, true cardiologist per-beat AAMI annotations, Lead I channel)")
lines.append(f"- RR Features: {RR_FEATURE_COUNT} causal (no rr_ratio, no prematurity)")
lines.append(f"- SV Head: dual sigmoid (V-head, S-head), real labels — no pseudo-labeling")
lines.append("")
lines.append("## 2. Primary Test Metrics")
lines.append(f"- Macro F1: {report['macro avg']['f1-score']:.4f}")
lines.append(f"- N: Recall={report['N']['recall']:.4f}, Precision={report['N']['precision']:.4f}, F1={report['N']['f1-score']:.4f}")
lines.append(f"- S: Recall={report['S']['recall']:.4f}, Precision={report['S']['precision']:.4f}, F1={report['S']['f1-score']:.4f}")
lines.append(f"- V: Recall={report['V']['recall']:.4f}, Precision={report['V']['precision']:.4f}, F1={report['V']['f1-score']:.4f}")
lines.append("")
lines.append("## 3. Cross-Database (external, never trained on)")
if mitdb_r: lines.append(f"- MIT-BIH (Lead II — domain mismatch expected): Macro F1={mitdb_r['macro avg']['f1-score']:.4f}, V Rec={mitdb_r['V']['recall']:.4f}, V Prec={mitdb_r['V']['precision']:.4f}")
if svdb_r: lines.append(f"- SVDB: Macro F1={svdb_r['macro avg']['f1-score']:.4f}, V Rec={svdb_r['V']['recall']:.4f}, V Prec={svdb_r['V']['precision']:.4f}")
else: lines.append("- SVDB: skipped (not required for primary results or quantization)")
lines.append("")
lines.append("## 4. Quantization")
lines.append(f"- Total: {(gs+ss)/1024:.1f} KB")
lines.append(f"- V MAE: {mae:.4f}, V mismatch: {mismatch:.3f}, S MAE: {s_mae:.4f}")
lines.append("")
lines.append("## 5. Thresholds")
lines.append(f"- Gate: {best_thr['gate']}, V: {best_thr['v']}, S: {best_thr.get('s', 'n/a')}")
lines.append("")
lines.append("## Limitations")
# STEP 7: EXACT 5-bullet list per user spec — do not paraphrase
lines.append("- INCART V/S source has only 32 unique patients.")
lines.append("- 3-class (N/S/V) only, not AAMI's full 5-class (F/Q not included).")
lines.append("- MIT-BIH cross-check is Lead II vs. Lead I training domain — a lower number there is expected, not a deployment concern.")
lines.append("- S class is intentionally deprioritized, not a bug.")
lines.append("- PTB-XL/CPSC's native PVC diagnostic statements exist but are record-level, not beat-level — noted as future work, not used in this version.")
report_text = "\n".join(lines)

with open(ROOT_OUT / "10_reports" / "FINAL_REPORT.md", "w", encoding="utf-8") as f:
    f.write(report_text)

print("="*80)
print("TARANG v15 FINAL — COMPLETE")
print("="*80)
print(report_text)
print(f"\nArtifacts: {ROOT_OUT}")


TARANG v15 FINAL — COMPLETE
# Tarang v15 — FINAL Report (Submission Baseline)
**Run ID:** 20260716_002728_72352b61
**Smoke Test:** False

## 1. Data
- N: PTB-XL + CPSC2018 (Lead I, NSR SNOMED-tagged records)
- V and S: INCART (12-lead, true cardiologist per-beat AAMI annotations, Lead I channel)
- RR Features: 4 causal (no rr_ratio, no prematurity)
- SV Head: dual sigmoid (V-head, S-head), real labels — no pseudo-labeling

## 2. Primary Test Metrics
- Macro F1: 0.6768
- N: Recall=0.9609, Precision=0.9993, F1=0.9797
- S: Recall=0.6810, Precision=0.1799, F1=0.2846
- V: Recall=0.9703, Precision=0.6328, F1=0.7660

## 3. Cross-Database (external, never trained on)
- MIT-BIH (Lead II — domain mismatch expected): Macro F1=0.4360, V Rec=0.2590, V Prec=0.1652
- SVDB: skipped (not required for primary results or quantization)

## 4. Quantization
- Total: 70.9 KB
- V MAE: 0.0038, V mismatch: 0.006, S MAE: 0.0043

## 5. Thresholds
- Gate: 0.25, V: 0.5000000000000001, S: 0.1

## Limitations
- INCAR

## 13. (Optional) CPSC REFERENCE.csv Footnote Check

Read-only informational check. Confirms whether the local CPSC2018 distribution uses native numeric PVC codes in `REFERENCE.csv` (the ICBEB2018 format) rather than SNOMED tags in `.hea` headers. **Does not retrain. Does not change labels. Does not affect any prior cell.** Output is appended as a footnote to the limitations section if PVC codes are present.


In [12]:
# STEP 8 (optional, read-only): CPSC REFERENCE.csv footnote check
# This cell runs ONLY after the final report (cell 22) has already been written.
# It cannot break anything upstream — it only reads a CSV and prints a footnote.
print("=== STEP 8 (optional): CPSC REFERENCE.csv read-only check ===")
try:
    ref_path = os.path.join(DATASET_PATHS['cpsc'], 'REFERENCE.csv')
    if not os.path.isfile(ref_path):
        print(f"  REFERENCE.csv not found at {ref_path} — skipping (no action taken)")
    else:
        ref = pd.read_csv(ref_path)
        print(f"  REFERENCE.csv loaded: {len(ref)} rows, columns = {ref.columns.tolist()}")
        # ICBEB2018 CPSC class codes: 1=N, 2=AF, 3=I-AVB, 4=LBBB, 5=RBBB, 6=PAC, 7=PVC, 8=STD, 9=STE
        # We only care about PVC (code 7) and PAC (code 6) for footnote purposes.
        code_col = ref.columns[1] if len(ref.columns) > 1 else ref.columns[0]
        vc = ref[code_col].value_counts().sort_index()
        print(f"  Class code value counts ({code_col}):")
        for code, count in vc.items():
            label_map = {1:'N', 2:'AF', 3:'I-AVB', 4:'LBBB', 5:'RBBB', 6:'PAC', 7:'PVC', 8:'STD', 9:'STE'}
            tag = label_map.get(int(code), '?') if str(code).isdigit() else '?'
            print(f"    code {code} ({tag}): {count} records")

        n_pvc = int(vc.get(7, 0))
        n_pac = int(vc.get(6, 0))
        if n_pvc > 0:
            footnote = (
                f"CPSC2018 REFERENCE.csv footnote: native PVC code (7) present on "
                f"{n_pvc} records, PAC code (6) on {n_pac} records. These are "
                f"record-level diagnostic statements, not beat-level annotations — "
                f"same weak-labeling limitation as PTB-XL. Noted as future work; "
                f"not integrated into v15 training (per hard rule: no new training "
                f"data sources before submission)."
            )
            print(f"\n  Footnote (appended to limitations):\n  {footnote}")
            # Append to FINAL_REPORT.md without rewriting it
            with open(ROOT_OUT / "10_reports" / "FINAL_REPORT.md", "a", encoding='utf-8') as f:
                f.write("\n\n## Limitations Footnote (CPSC REFERENCE.csv check)\n\n")
                f.write(footnote + "\n")
            print(f"  → Appended to {ROOT_OUT / '10_reports' / 'FINAL_REPORT.md'}")
        else:
            print("  No PVC codes (7) found in REFERENCE.csv — no footnote to add.")
            print("  (Confirms the earlier zero-PVC finding; CPSC distribution here uses SNOMED-in-.hea format.)")
except Exception as e:
    print(f"  CPSC REFERENCE.csv check failed cleanly (non-blocking): {type(e).__name__}: {str(e)[:120]}")
    print("  Continuing — this step is optional and does not affect any prior result.")
print("=== END STEP 8 ===")


=== STEP 8 (optional): CPSC REFERENCE.csv read-only check ===
  REFERENCE.csv not found at C:/MMD Public/Hackathons/Team Ocelleon/dataset\CPSC2018\REFERENCE.csv — skipping (no action taken)
=== END STEP 8 ===
